In [33]:
import pandas as pd
import sklearn.model_selection as cv
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import (
    SVC,
)
from statsmodels.stats.contingency_tables import mcnemar

We load the dataset and prepare (not in this example, but add the preprocessing here) it for feeding algorithms.

In [34]:
url = "https://archive.ics.uci.edu/ml/machine-learning-databases/ionosphere/ionosphere.data"
df = pd.read_csv(url, header = None)
X = df.drop([34], axis = 1).values
y = df[34].values

(X_train, X_test,  y_train, y_test) = cv.train_test_split(
    X, 
    y, 
    test_size = .3, 
    stratify = y,
    random_state = 1
)

Lets obtain two models (A and B) and compare them using the McNemar test:

In [15]:
clf = GaussianNB()
prediction_model_A = clf.fit(X_train, y_train).predict(X_test)

knc = SVC(kernel = "linear")
prediction_model_B = knc.fit(X_train, y_train).predict(X_test)

In [39]:
# [
#   [Number of items misclassified by both A and B, Number of items misclassified by A but not by B ],
#   [Number of items misclassified by B but not by A, Number of items classified correctly by both A and B]
# ]
both_right = [
    (y_test[i] == prediction_model_A[i]) and (y_test[i] == prediction_model_B[i]) 
    for i in range(len(y_test))
]

only_B_right = [
    (y_test[i] != prediction_model_A[i]) and (y_test[i] == prediction_model_B[i])
    for i in range(len(y_test))
]

only_A_right = [
    (y_test[i] == prediction_model_A[i]) and (y_test[i] != prediction_model_B[i])
    for i in range(len(y_test))
]

both_wrong = [
    (y_test[i] != prediction_model_A[i]) and (y_test[i] != prediction_model_B[i])
    for i in range(len(y_test))
]

In [40]:
# Compute the contingency table
contingency_table = [
    [sum(both_right), sum(only_B_right)],
    [sum(only_A_right), sum(both_wrong)]
]
print(f"Contingency table: {contingency_table}")

significance_value = 0.05

# McNemar's Test with the continuity correction
results_mcnemar = mcnemar(contingency_table, exact=False, correction=True)
print(f"Statistics value: {results_mcnemar.statistic}\np-value: {results_mcnemar.pvalue}")


if results_mcnemar.pvalue < significance_value:
  print("Reject Null hypothesis")
else:
  print("Fail to reject Null hypothesis -> we can not distinguish both models!")

Contingency table: [[83, 6], [10, 7]]
Statistics value: 0.5625
p-value: 0.4532547047537363
Fail to reject Null hypotesis -> we can not distinguish both models!
